# 08 — Threshold tuning

> **Dữ liệu synthetic:** threshold được chọn trên validation, sau đó đóng băng trước khi đánh giá test.

Credit dùng expected class score `Σ k × p(k)` và ngưỡng tối đa QWK. Fraud dùng cutoff phân vị 95% để ưu tiên top 5% hồ sơ review.

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    raise FileNotFoundError('Không tìm thấy configs/config.yaml. Hãy chỉnh PROJECT_ROOT.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.threshold import expected_class_score, find_qwk_thresholds, fraud_top_k_metrics
from src.models.predict import load_model_bundle
from src.utils.config import load_config
from src.utils.io import read_parquet_safe

## Fit policy on validation

Cell này chỉ đọc validation và có thể tái tạo policy đã lưu; không ghi đè artifact hoặc đọc test.

In [ ]:
config = load_config(PROJECT_ROOT / 'configs/config.yaml')
feature_dir = PROJECT_ROOT / config['features']['output_dir']
X_valid = read_parquet_safe(feature_dir / 'X_valid.parquet')
y_credit_valid = read_parquet_safe(feature_dir / 'y_credit_valid.parquet').iloc[:, 0]
y_fraud_valid = read_parquet_safe(feature_dir / 'y_fraud_valid.parquet').iloc[:, 0]
credit_bundle = load_model_bundle('v1.0.0', 'credit_risk', PROJECT_ROOT / 'models')
fraud_bundle = load_model_bundle('v1.0.0', 'fraud_risk_score', PROJECT_ROOT / 'models')
credit_proba = credit_bundle.model.predict(X_valid.to_numpy(dtype=float))
fraud_scores = fraud_bundle.model.predict(X_valid.to_numpy(dtype=float))
credit_policy = find_qwk_thresholds(y_credit_valid, credit_proba, n_classes=len(credit_bundle.metadata['classes']), candidate_quantiles=config['governance']['threshold_candidate_quantiles'])
fraud_policy = fraud_top_k_metrics(y_fraud_valid, fraud_scores, k=0.05)
pd.DataFrame([
    {'task': 'credit_risk', 'qwk_argmax': credit_policy.qwk_argmax, 'qwk_thresholded': credit_policy.qwk_thresholded, 'thresholds': credit_policy.thresholds},
    {'task': 'fraud_risk_score', **fraud_policy},
])

In [ ]:
scores = expected_class_score(credit_proba)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scores, bins=60, color='#2563eb', alpha=0.8)
for threshold in credit_policy.thresholds:
    ax.axvline(threshold, color='#d97706', linestyle='--')
ax.set(xlabel='Expected credit-risk class score', ylabel='Customers', title='Validation score distribution and QWK thresholds')
plt.show()

## Frozen test result

Test đã được đánh giá đúng một lần. Cell dưới chỉ đọc metric đã lưu, không predict hay tune lại trên test.

In [ ]:
credit_test = json.loads((PROJECT_ROOT / 'models/credit_risk/v1.0.0/threshold_test_metrics.json').read_text(encoding='utf-8'))
fraud_test = json.loads((PROJECT_ROOT / 'models/fraud_risk/v1.0.0/threshold_test_metrics.json').read_text(encoding='utf-8'))
pd.DataFrame([
    {'task': 'credit_risk', 'qwk': credit_test['qwk'], 'macro_f1': credit_test['macro_f1']},
    {'task': 'fraud_risk_score', 'rmse': fraud_test['rmse'], 'mae': fraud_test['mae'], 'r2': fraud_test['r2'], 'precision_at_k': fraud_test['precision_at_k']},
])

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=str(PROJECT_ROOT / 'reports/threshold_report.html'), width='100%', height=700))

## Regenerate validation artifact

```powershell
python -m scripts.run_threshold_report
```

Không dùng `--final`: test đã bị khóa để tránh đánh giá lặp lại.